In [212]:
import torch
import triton
import triton.language as tl

In [213]:
def add_torch(x, const_val):
    return x + const_val


@triton.jit
def add_kernel(x_ptr, out_ptr, const_val, n_elements: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)  # все элементы
    x = tl.load(x_ptr + offsets)
    output = x + const_val
    tl.store(out_ptr + offsets, output)


def add_triton(x, const_val):
    n_elements =  x.numel()
    out = torch.empty_like(x)
    add_kernel[(1,)](x_ptr=x, out_ptr=out, const_val=const_val, n_elements=n_elements,  BLOCK_SIZE=n_elements)
    return out


x = torch.tensor([1, 2, 3, 4], dtype=torch.float32, device='cuda')
const_val = 10.0
torch_res = add_torch(x, const_val)
triton_res = add_triton(x, const_val)

print("Torch result:", torch_res)
print("Triton result:", triton_res)
assert torch.allclose(torch_res, triton_res), "Results do not match!"

Torch result: tensor([11., 12., 13., 14.], device='cuda:0')
Triton result: tensor([11., 12., 13., 14.], device='cuda:0')


In [214]:

def add_torch(x, const_val):
    return x + const_val

@triton.jit
def add_kernel(x_ptr, out_ptr, const_val, n_elements: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask)
    output = x + const_val
    tl.store(out_ptr + offsets, output, mask)


def add_triton(x, const_val):
    n_elements =  x.numel()
    out = torch.empty_like(x)
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    add_kernel[grid](x_ptr=x, out_ptr=out, const_val=const_val, n_elements=n_elements, BLOCK_SIZE=4)
    return out

x = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32, device='cuda')
const_val = 10.0
torch_res = add_torch(x, const_val)
triton_res = add_triton(x, const_val)

print("Torch result:", torch_res)
print("Triton result:", triton_res)
assert torch.allclose(torch_res, triton_res), "Results do not match!"

Torch result: tensor([11., 12., 13., 14., 15.], device='cuda:0')
Triton result: tensor([11., 12., 13., 14., 15.], device='cuda:0')


In [215]:
def add_vec_torch(x, y):
    return x[None, :] + y[:, None]  # (N1, N0)

def add_vec_triton(x, y):
    # Use two BLOCK constexprs and map (row, col) with arange.
    n, m = x.numel(), y.numel()
    res_tensor = torch.empty((m, n), dtype=torch.float32, device='cuda')  
    
    add_vec_triton_kernel[(1,)](x, y, res_tensor, m, n)
    return res_tensor
    
    
@triton.jit
def add_vec_triton_kernel(x_ptr, y_ptr, res_ptr,  B0: tl.constexpr, B1: tl.constexpr):
    off_y = tl.arange(0, B0)  
    off_x = tl.arange(0, B1)  

    x = tl.load(x_ptr + off_x)[None, :]  
    y = tl.load(y_ptr + off_y)[:, None]  

    result = x + y

    tl.store(res_ptr + off_y[:, None] * B1 + off_x[None, :], result)



x = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8], dtype=torch.float32).cuda()  
y = torch.tensor([10, 20, 30, 40], dtype=torch.float32).cuda()
print(add_vec_triton(x, y))
assert torch.allclose(add_vec_torch(x, y), add_vec_triton(x, y))

tensor([[11., 12., 13., 14., 15., 16., 17., 18.],
        [21., 22., 23., 24., 25., 26., 27., 28.],
        [31., 32., 33., 34., 35., 36., 37., 38.],
        [41., 42., 43., 44., 45., 46., 47., 48.]], device='cuda:0')


In [216]:
def add_vec_torch(x, y):
    return x[None, :] + y[:, None]  # (N1, N0)

def add_vec_triton(x, y, B0=2, B1=2):
    # Use two BLOCK constexprs and map (row, col) with arange.
    n, m = x.numel(), y.numel()
    res_tensor = torch.empty((m, n), dtype=torch.float32, device='cuda')
    grid = lambda meta: (triton.cdiv(m, meta['B0']), triton.cdiv(n, meta['B1']))
    add_vec_triton_kernel[grid](x, y, res_tensor, m, n, B0, B1)
    return res_tensor
    
    
@triton.jit
def add_vec_triton_kernel(x_ptr, y_ptr, res_ptr,  m, n, B0: tl.constexpr, B1: tl.constexpr):
    pid0 = tl.program_id(axis=0)
    pid1 = tl.program_id(axis=1)
    off_x = tl.arange(0, B1)  + pid1*B1
    off_y = tl.arange(0, B0) + pid0*B0  
    mask_x = off_x < n
    mask_y = off_y < m
    mask = mask_x[None, :] & mask_y[:, None]

    x = tl.load(x_ptr + off_x, mask_x, 0)[None, :]  
    y = tl.load(y_ptr + off_y, mask_y, 0)[:, None]  

    result = x + y

    tl.store(res_ptr + off_y[:, None] * n + off_x[None, :], result, mask)



x = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8], dtype=torch.float32).cuda()  
y = torch.tensor([10, 20, 30, 40], dtype=torch.float32).cuda()
print(add_vec_triton(x, y))
assert torch.allclose(add_vec_torch(x, y), add_vec_triton(x, y))

tensor([[11., 12., 13., 14., 15., 16., 17., 18.],
        [21., 22., 23., 24., 25., 26., 27., 28.],
        [31., 32., 33., 34., 35., 36., 37., 38.],
        [41., 42., 43., 44., 45., 46., 47., 48.]], device='cuda:0')


In [217]:
 tl.constexpr(0)

constexpr[0]

In [218]:
def mul_relu_block_torch(x, y):
     return torch.relu(x[None, :] * y[:, None])

def mul_relu_block_triton(x, y, B0=2, B1=2):
    # Use two BLOCK constexprs and map (row, col) with arange.
    n, m = x.numel(), y.numel()
    res_tensor = torch.empty((m, n), dtype=torch.float32, device='cuda')
    grid = lambda meta: (triton.cdiv(m, meta['B0']), triton.cdiv(n, meta['B1']))
    mul_relu_block_triton_kernel[grid](x, y, res_tensor, m, n, B0, B1)
    return res_tensor
    
    
@triton.jit
def mul_relu_block_triton_kernel(x_ptr, y_ptr, res_ptr,  m, n, B0: tl.constexpr, B1: tl.constexpr):
    pid0 = tl.program_id(axis=0)
    pid1 = tl.program_id(axis=1)
    off_x = tl.arange(0, B1)  + pid1*B1
    off_y = tl.arange(0, B0) + pid0*B0  
    mask_x = off_x < n
    mask_y = off_y < m
    mask = mask_x[None, :] & mask_y[:, None]

    x = tl.load(x_ptr + off_x, mask_x, 0)[None, :]  
    y = tl.load(y_ptr + off_y, mask_y, 0)[:, None]  

    result = x * y
    result = tl.maximum(result, 0.0)

    tl.store(res_ptr + off_y[:, None] * n + off_x[None, :], result, mask)



x = torch.tensor([1, 2, 3, -4, 5, 6, 7, 8], dtype=torch.float32).cuda()  
y = torch.tensor([1, 2, 3, 4], dtype=torch.float32).cuda()
print( mul_relu_block_triton(x, y))
assert torch.allclose(mul_relu_block_torch(x, y), mul_relu_block_triton(x, y))

tensor([[ 1.,  2.,  3.,  0.,  5.,  6.,  7.,  8.],
        [ 2.,  4.,  6.,  0., 10., 12., 14., 16.],
        [ 3.,  6.,  9.,  0., 15., 18., 21., 24.],
        [ 4.,  8., 12.,  0., 20., 24., 28., 32.]], device='cuda:0')


In [219]:
def mul_relu_block_back_torch(x, y, dz):
    x = x.clone().requires_grad_(True)
    y = y.clone().requires_grad_(True)
    z = torch.relu(x * y[:, None])
    z.backward(dz)
    dx = x.grad
    return dx

@triton.jit
def mul_relu_block_back_kernel(x_ptr, y_ptr, dz_ptr, dx_ptr, n, m, B0: tl.constexpr, B1: tl.constexpr):
    pid0 = tl.program_id(0)  
    pid1 = tl.program_id(1)  

    off_x = pid1 * B1 + tl.arange(0, B1) 
    off_y = pid0 * B0 + tl.arange(0, B0)  

    mask_x = off_x < n
    mask_y = off_y < m

    x = tl.load(x_ptr + off_x, mask_x, 0.0)[None, :]
    y = tl.load(y_ptr + off_y, mask_y, 0.0)[:, None]   
    dz = tl.load(dz_ptr + off_y[:, None] * n + off_x[None, :], mask_y[:, None] & mask_x[None, :], 0.0)

    # u = x * y
    u = x * y
    grad_mask = u > 0
    contrib = grad_mask * y * dz  
    # partial reduction over y dimension
    dx_local = tl.sum(contrib, axis=0)

    tl.atomic_add(dx_ptr + off_x, dx_local, mask_x)


def mul_relu_block_back_triton(x, y, dz, B0=4, B1=4):
    n, m = x.numel(), y.numel()
    dx = torch.zeros_like(x)
    grid = lambda meta: (triton.cdiv(m, meta['B0']), triton.cdiv(n, meta['B1']))
    mul_relu_block_back_kernel[grid](x, y, dz, dx, n, m, B0, B1)
    return dx

x = torch.tensor([-1, 2, -3, 4, 5, -6], dtype=torch.float32, device='cuda')
y = torch.tensor([1, -2, 3, -4], dtype=torch.float32, device='cuda')
dz = torch.randn((y.numel(), x.numel()), dtype=torch.float32, device='cuda')

dx_torch = mul_relu_block_back_torch(x, y, dz)
dx_triton = mul_relu_block_back_triton(x, y, dz)

print(dx_triton)
assert torch.allclose(dx_torch, dx_triton, rtol=1e-4, atol=1e-5)


tensor([ 5.6186, -2.4557,  4.7473,  3.0607,  0.1319, -4.6458], device='cuda:0')


In [224]:
def sum_torch(x):
    return x.sum(1)


@triton.jit
def sum_triton_kernel(x_ptr, y_ptr, N0, T: tl.constexpr, B1: tl.constexpr):
    pid = tl.program_id(0)
    row_start = pid * T
    offsets = tl.constexpr(0.0)       # 1D tile offsets
    
    acc = tl.zeros((), dtype=tl.float32)  # scalar

    for t in tl.static_range(0, T, B1):
        idx = row_start + t + offsets
        mask = (t + offsets) < T
        x = tl.load(x_ptr + idx, mask=mask, other=0.0)
        acc += tl.sum(x, axis=0)       
    tl.store(y_ptr + pid, acc)




def sum_triton(x, B1=8):
    N0, T = x.shape
    y = torch.empty((N0,), dtype=torch.float32, device='cuda')
    grid = lambda meta: (N0,)
    sum_rows_kernel[grid](x, y, N0, T=T, B1=B1)
    return y


    
x = torch.randn((4, 70), dtype=torch.float32, device='cuda')


print(sum_triton(x))
assert torch.allclose(y_torch, y_triton)

tensor([  5.7804,  -6.5779,   0.7865, -13.2327], device='cuda:0')
